# Advanced Machine Learning – Final Test (Group B)
## 2025-06-06


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from matplotlib.backends.backend_pdf import PdfPages

---
# Task 1 – Nonparametric Regression (10 pts)


In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('weights.csv')
# Assume columns are named 'x' and 'y'; adjust if needed
x = df.iloc[:, 0].values.astype(float)
y = df.iloc[:, 1].values.astype(float)
n = len(x)
print(f'Loaded {n} observations.')

In [ ]:
# ── 4-fold split (random, reproducible) ────────────────────────────────────────
np.random.seed(42)
idx = np.random.permutation(n)
folds = np.array_split(idx, 4)   # list of 4 index arrays

In [ ]:
# ── Nadaraya-Watson helpers ─────────────────────────────────────────────────────

def gaussian_kernel(u):
    """Standard Gaussian kernel K(u) = (1/sqrt(2pi)) * exp(-u^2/2)."""
    return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)


def nw_predict(x_train, y_train, x_test, h):
    """
    Nadaraya-Watson kernel regression prediction.
    For each test point x*, compute:
        f_hat(x*) = sum_i K((x*-x_i)/h) * y_i  /  sum_i K((x*-x_i)/h)
    """
    preds = np.zeros(len(x_test))
    for j, xj in enumerate(x_test):
        w = gaussian_kernel((xj - x_train) / h)
        denom = w.sum()
        if denom == 0:
            preds[j] = np.mean(y_train)   # fallback for extreme h
        else:
            preds[j] = (w * y_train).sum() / denom
    return preds

In [ ]:
# ── 4-fold CV for Nadaraya-Watson ───────────────────────────────────────────────

# Bandwidth grid: log-spaced between ~0.5 and ~50 (adjust to data scale)
bandwidths = np.logspace(-0.5, 2.5, 40)

nw_cv_mse = []
for h in bandwidths:
    fold_mses = []
    for k in range(4):
        val_idx   = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(4) if j != k])
        y_pred = nw_predict(x[train_idx], y[train_idx], x[val_idx], h)
        fold_mses.append(np.mean((y[val_idx] - y_pred)**2))
    nw_cv_mse.append(np.mean(fold_mses))

nw_cv_mse = np.array(nw_cv_mse)
h_star = bandwidths[np.argmin(nw_cv_mse)]
print(f'Optimal bandwidth h* = {h_star:.4f}')

In [ ]:
# ── Plot MSE vs h (Nadaraya-Watson) ────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(bandwidths, nw_cv_mse, marker='o', markersize=3)
plt.axvline(h_star, color='red', linestyle='--', label=f'h* = {h_star:.3f}')
plt.xscale('log')
plt.xlabel('Bandwidth h')
plt.ylabel('CV-MSE')
plt.title('Nadaraya-Watson: 4-fold CV-MSE vs bandwidth')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Smoothing-spline helper ─────────────────────────────────────────────────────
# scipy's UnivariateSpline uses a smoothing factor 's' (sum of squared residuals).
# We map lambda (regularisation weight) -> s = lambda * n  (proportional scaling).

def ss_predict(x_train, y_train, x_test, lam):
    """
    Fit a smoothing spline with regularisation parameter lambda on (x_train, y_train)
    and predict at x_test points.
    """
    # sort by x – required by UnivariateSpline
    order = np.argsort(x_train)
    spl = UnivariateSpline(x_train[order], y_train[order],
                           s=lam * len(x_train), k=3)
    return spl(x_test)

In [ ]:
# ── 4-fold CV for smoothing splines ────────────────────────────────────────────

lambdas = np.logspace(-2, 4, 40)   # wide range; adjust based on data scale

ss_cv_mse = []
for lam in lambdas:
    fold_mses = []
    for k in range(4):
        val_idx   = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(4) if j != k])
        try:
            y_pred = ss_predict(x[train_idx], y[train_idx], x[val_idx], lam)
            fold_mses.append(np.mean((y[val_idx] - y_pred)**2))
        except Exception:
            fold_mses.append(np.inf)   # spline fitting failed for this lambda
    ss_cv_mse.append(np.mean(fold_mses))

ss_cv_mse = np.array(ss_cv_mse)
lam_star = lambdas[np.argmin(ss_cv_mse)]
print(f'Optimal smoothing parameter lambda* = {lam_star:.4f}')

In [ ]:
# ── Plot MSE vs lambda (smoothing splines) ──────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(lambdas, ss_cv_mse, marker='o', markersize=3, color='green')
plt.axvline(lam_star, color='red', linestyle='--', label=f'λ* = {lam_star:.3f}')
plt.xscale('log')
plt.xlabel('Smoothing parameter λ')
plt.ylabel('CV-MSE')
plt.title('Smoothing Splines: 4-fold CV-MSE vs λ')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Final fits on all data + overlay plot ──────────────────────────────────────
x_grid = np.linspace(x.min(), x.max(), 500)

# Nadaraya-Watson with h*
y_nw = nw_predict(x, y, x_grid, h_star)

# Smoothing spline with lambda*
order = np.argsort(x)
spl_final = UnivariateSpline(x[order], y[order], s=lam_star * n, k=3)
y_ss = spl_final(x_grid)

plt.figure(figsize=(10, 5))
plt.scatter(x, y, alpha=0.4, s=20, label='Data', color='steelblue')
plt.plot(x_grid, y_nw, color='red',   lw=2, label=f'NW (h*={h_star:.2f})')
plt.plot(x_grid, y_ss, color='green', lw=2, label=f'SS (λ*={lam_star:.2f})')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Nadaraya-Watson vs Smoothing Spline – fitted curves')
plt.legend()
plt.tight_layout()
plt.show()

---
# Task 2 – Generate Artificial Data (4 pts)


In [ ]:
np.random.seed(0)

def generate_data(n=1000):
    """
    Generate n samples:
    - X1 ~ 0.25*N(-1,0.2) + 0.25*N(1,0.2) + 0.5*N(5,0.2)  (mixture of 3 Gaussians)
    - X2,...,X10 ~ N(0,1)
    - Y | X: P(Y=1|X) = 0.1 if X1 < 3, else 0.9
    """
    # ── X1: mixture of 3 Gaussians ──────────────────────────────────────────────
    component = np.random.choice([0, 1, 2], size=n, p=[0.25, 0.25, 0.50])
    means = [-1, 1, 5]
    X1 = np.array([np.random.normal(means[c], 0.2) for c in component])

    # ── X2..X10: standard normal ────────────────────────────────────────────────
    X_rest = np.random.normal(0, 1, size=(n, 9))

    X = np.column_stack([X1, X_rest])   # shape (n, 10)

    # ── Binary labels with class-conditional probability ────────────────────────
    prob = np.where(X1 < 3, 0.1, 0.9)
    Y = np.random.binomial(1, prob)

    return X, Y


X_train, Y_train = generate_data(1000)
X_test,  Y_test  = generate_data(1000)

print('Train class distribution:', np.bincount(Y_train))
print('Test  class distribution:', np.bincount(Y_test))

---
# Task 3 – Weighted Logistic Regression via Gradient Descent (16 pts)


In [ ]:
# ── Mathematical derivation (written in comment) ────────────────────────────────
#
# Risk function:
#   R(θ) = -(1/n) * Σ_i ||x_i||_∞ * [ y_i * x_i^T θ + log(1 - σ(x_i^T θ)) ]
#          + 0.01 * ||θ||_2^2
#
# Let  a_i = x_i^T θ,  w_i = ||x_i||_∞,  σ_i = σ(a_i) = 1/(1+exp(-a_i))
#
# Note: log(1 - σ(a)) = log(σ(-a)) = -log(1+exp(a))   (log-sum-exp form)
#
# Derivative of the i-th term w.r.t. θ:
#   d/dθ [ y_i * a_i + log(1 - σ(a_i)) ]
#   = y_i * x_i  +  d/dθ log(1 - σ(a_i))
#   = y_i * x_i  -  σ(a_i) * x_i
#   = (y_i - σ(a_i)) * x_i
#
# Therefore:
#   ∇R(θ) = -(1/n) * Σ_i w_i * (y_i - σ_i) * x_i   +   2 * 0.01 * θ
#
# In matrix form (X: n×d, w: n-vector, σ: n-vector):
#   ∇R(θ) = -(1/n) * X^T (w ⊙ (y - σ))  +  0.02 * θ

def sigmoid(z):
    """Numerically stable sigmoid: σ(z) = 1/(1+exp(-z))."""
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))


def compute_risk(X, Y, theta):
    """
    Evaluate R(θ) on dataset (X, Y).
    w_i = ||x_i||_∞  (L-inf norm of each row)
    """
    n = len(Y)
    w = np.max(np.abs(X), axis=1)       # L-inf norm per sample
    a = X @ theta                        # linear scores
    sig = sigmoid(a)
    # log(1 - σ(a)) = -log(1 + exp(a)); use numerically stable form
    log1msig = -np.log1p(np.exp(np.clip(a, -500, 500)))   # clip to avoid overflow
    nll = -(1/n) * np.sum(w * (Y * a + log1msig))
    reg = 0.01 * np.dot(theta, theta)
    return nll + reg


def compute_gradient(X, Y, theta):
    """
    Compute ∇R(θ):
        -(1/n) * X^T (w ⊙ (Y - σ(Xθ)))  +  0.02 * θ
    """
    n = len(Y)
    w = np.max(np.abs(X), axis=1)       # L-inf norm per sample
    sig = sigmoid(X @ theta)
    residuals = w * (Y - sig)            # element-wise: w_i * (y_i - σ_i)
    grad = -(1/n) * (X.T @ residuals) + 0.02 * theta
    return grad


def predict(X, theta):
    """Predict class labels: ŷ = 1 if σ(x^T θ) > 0.5, else 0."""
    return (sigmoid(X @ theta) > 0.5).astype(int)

In [ ]:
# ── Gradient Descent ────────────────────────────────────────────────────────────
lr      = 0.01
epochs  = 100
d       = X_train.shape[1]              # feature dimensionality = 10
theta   = np.zeros(d)                   # initialisation at zero vector

risk_history       = []
train_acc_history  = []
test_acc_history   = []

# Record metrics at iteration 0 (initialisation point)
risk_history.append(compute_risk(X_train, Y_train, theta))
train_acc_history.append(np.mean(predict(X_train, theta) == Y_train))
test_acc_history.append(np.mean(predict(X_test,  theta) == Y_test))

for epoch in range(1, epochs + 1):
    grad  = compute_gradient(X_train, Y_train, theta)
    theta = theta - lr * grad            # gradient descent step

    risk_history.append(compute_risk(X_train, Y_train, theta))
    train_acc_history.append(np.mean(predict(X_train, theta) == Y_train))
    test_acc_history.append(np.mean(predict(X_test,  theta)  == Y_test))

print(f'Final risk:       {risk_history[-1]:.4f}')
print(f'Final train acc:  {train_acc_history[-1]:.4f}')
print(f'Final test  acc:  {test_acc_history[-1]:.4f}')

In [ ]:
# ── RISK.pdf ────────────────────────────────────────────────────────────────────
iters = np.arange(len(risk_history))   # 0..100

with PdfPages('RISK.pdf') as pdf:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(iters, risk_history, color='steelblue', lw=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('R(θ)')
    ax.set_title('Risk R(θ) over GD iterations (incl. init at iter 0)')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    pdf.savefig(fig)
    plt.show()

print('Saved RISK.pdf')

In [ ]:
# ── TRAIN.pdf ───────────────────────────────────────────────────────────────────
with PdfPages('TRAIN.pdf') as pdf:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(iters, train_acc_history, color='green', lw=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training accuracy over GD iterations (incl. init at iter 0)')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    pdf.savefig(fig)
    plt.show()

print('Saved TRAIN.pdf')

In [ ]:
# ── TEST.pdf ────────────────────────────────────────────────────────────────────
with PdfPages('TEST.pdf') as pdf:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(iters, test_acc_history, color='red', lw=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Accuracy')
    ax.set_title('Test accuracy over GD iterations (incl. init at iter 0)')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    pdf.savefig(fig)
    plt.show()

print('Saved TEST.pdf')